In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_rscf import lps_solver

In [2]:
csv_file = 'closed_shell_atoms_vs_rhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 8 rows found.


In [11]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}

METHOD = "TF0.166666W PBEx"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.166666
EXC = ['GGA_X_PBE', 1.0, 'LDA_C_VWN', 0.0]
FA = [False, 1.0]
DIIS = True
MAX_ITER = 14000
DAMPING = [0.99, 0.9, 0.0001]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "ChemPot,Ha": round(mu, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Skipping He (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Be (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Ne (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Mg (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Ar (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Ca (Already exists for TF0.166666W PBEx/UGBS_S)
Skipping Zn (Already exists for TF0.166666W PBEx/UGBS_S)
Calculating Kr with TF0.166666W PBEx...
Number of basis functions:   31

Starting SCF iterations:

    Iter               Energy         ChemPot       Delta E         dRMS

SCF Iter  1:    117947.11678599   -6.48000E+02    1.17947E+05    8.26862E+03
SCF Iter  2:    115653.92898659    5.08789E-02   -2.29319E+03    8.12150E+03
SCF Iter  3:    113402.63854646   -3.49895E-02   -2.25129E+03    7.97694E+03
SCF Iter  4:    140718.06900516   -1.07445E+06    2.73154E+04    1.01261E+04
SCF Iter  5:    171846.49695782   -1.07525E+06    3.11284E+04    1.15199E+04
SCF Iter  6:    208795.83361024   -1.07060E

In [12]:
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha","ChemPot,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFD0.166666W,He,UGBS_S,6,1000,-2.951247,-0.067106,185,True,0.90,0.9,0.0010
1,TFD0.166666W,Be,UGBS_S,6,1000,-15.040391,-0.069849,233,True,0.90,0.9,0.0010
2,TFD0.166666W,Ne,UGBS_S,6,1000,-132.508856,-0.072529,827,True,0.90,0.9,0.0010
3,TFD0.166666W,Mg,UGBS_S,6,1000,-204.540690,-0.072963,1732,True,0.90,0.9,0.0010
4,TFD0.166666W,Ar,UGBS_S,6,1000,-537.208760,-0.073833,2513,True,0.90,0.9,0.0010
5,TFD0.166666W,Ca,UGBS_S,6,1000,-690.396266,-0.074040,3724,True,0.90,0.9,0.0010
6,TFD0.166666W,Zn,UGBS_S,6,1000,-1812.192490,-0.074767,7974,True,0.99,0.9,0.0001
7,TFD0.166666W,Kr,UGBS_S,6,1000,-2795.914635,-0.075063,8692,True,0.99,0.9,0.0001
8,TF0.166666W PBEx,He,UGBS_S,6,1000,-3.097069,-0.081193,238,True,0.90,0.9,0.0010
9,TF0.166666W PBEx,Be,UGBS_S,6,1000,-15.388877,-0.081756,348,True,0.90,0.9,0.0010


In [13]:
df.to_csv(csv_file, index=False)